We'll **use DistilBERT** transformer (because it's smaller than BERT but the difference in the result will not be that big)

# Step 1: Data Prep

Decision 1: we'll use **raw text (not `final_message`)** because transformers can use punctuation, word order, capitalization.

Decision 2: for a correct comparison we'll use **the same training set** as baseline.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

# just the raw message strings and the label
df = pd.read_csv('../data/SMSSpamCollection', sep='\t', header=None, names=['label', 'message'])
df = df.drop_duplicates()

y = df['label'].map({'ham': 0, 'spam': 1})

train_texts, test_texts, y_train, y_test = train_test_split(
    df['message'], y, test_size=0.2, random_state=42, stratify=y
)

# Step 2: Tokenization

DistillBURT uses **WordPiece tokenization**. It split strings into **subword units**, not just by spaces like in the baseline. Common whole words stay as a token, rare words are broken into tokens.

It's important because our dataset have a lot of messy real-world text message. Word-count-based methods (our baseline) do not handle well with this.

In [2]:
from transformers import DistilBertTokenizer

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# quick look at how it tokenizes one message
sample = train_texts.iloc[0]
tokens = tokenizer.tokenize(sample)
print(sample)
print(tokens)

Ta-Daaaaa! I am home babe, are you still up ?
['ta', '-', 'da', '##aa', '##aa', '!', 'i', 'am', 'home', 'babe', ',', 'are', 'you', 'still', 'up', '?']


/home/philip/miniforge3/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


We used most-used `uncased` for this project (and losing caps signal unlike baseline model).

# Step 3: Padding and Truncation

Batched matices requires every row to be the same length.

* **Padding**: special `[PAD]` token adding at the end of short message until they reach a target length'
* **Truncation**: messages longer than a target length get cut off.

To save computing resources, it is used **attention mask**: a second sequence of 1s (for real tokens) and 0s (for padding).

## Chosing the target length

In [3]:
# check how long messages actually are, in tokens, before picking a fixed length
token_lengths = train_texts.apply(lambda x: len(tokenizer.tokenize(x)))
print(token_lengths.describe())

count    4135.000000
mean       23.000967
std        16.949354
min         1.000000
25%        11.000000
50%        17.000000
75%        33.000000
max       219.000000
Name: message, dtype: float64


In [4]:
token_lengths.quantile(0.95)

np.float64(52.0)

* Half of messages are <= 17 tokens.
* 95% of messages are <= 52 tokens.

For example:
* use 219 => wasted computation.
* use 20 => cut off meaningful chunk => lose context.

Then we use length that covers the large majority 95% (the standard practice) and round a clean number powers of 2 for efficiency.

In [5]:
# Split text into tokens and converts each token into its corresponding integer ID
# and add padding and build an attention mask 

train_encodings = tokenizer(
    list(train_texts),
    truncation=True,
    padding=True,
    max_length=64,
    return_tensors='pt'  # returns PyTorch tensor
)

test_encodings = tokenizer(
    list(test_texts),
    truncation=True,
    padding=True,
    max_length=64,
    return_tensors='pt'  # returns PyTorch tensor
)

In [6]:
train_encodings['input_ids'].shape

torch.Size([4135, 64])

So we have 4135 rows (for every message) and 64 columns for each token with padding (after truncating).

In [7]:
train_encodings['attention_mask'].shape

torch.Size([4135, 64])

The same size for an attention mask.

**Confirming padding:**

In [8]:
print(train_encodings['input_ids'][0])
print(train_encodings['attention_mask'][0])

tensor([  101, 11937,  1011,  4830, 11057, 11057,   999,  1045,  2572,  2188,
        11561,  1010,  2024,  2017,  2145,  2039,  1029,   102,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0])
tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])


That visual match confirms attention mechanism.

# Step 4: Wrapping into a PyTorch Dataset & Loading the Dataset

Now we create a custom Dataset class because Pytorch expect object that has:
* `__len__` - number of examples
* `__getitem__` - return an object by index

In [9]:
import torch

class SpamDataset(torch.utils.data.Dataset):  # inherits from PyTorch's Dataset class
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels.iloc[idx])  # 'labels' name matters for HuggingFace
        return item

train_dataset = SpamDataset(train_encodings, y_train)
test_dataset = SpamDataset(test_encodings, y_test)

In [10]:
# Loading the pretrained model itself
from transformers import DistilBertForSequenceClassification

# An additional layer on top of DistilBERT's core
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', num_labels=2  # 2 categories of output
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


The message above isn't a bug. The language understanding is ready (pretrained on original DistilBERT dataset) but the decision-making layer on top is untrained (needs fine-tuning).

# Step 5: Fine-tuning

We'll use HuggingFace Trainer API. It trains model for me (the batch-by-batch forward pass, loss calculation, backpropagation, weight updates).

In [11]:
from transformers import TrainingArguments, Trainer
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
import numpy as np

# We need to explicitly add metrics to Trainer because by default Trainer only tracks loss during training
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'precision': precision, 'recall': recall, 'f1': f1}

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,  # avoid memorizing
    per_device_train_batch_size=16,  # CPU-friendly choice
    per_device_eval_batch_size=16,
    eval_strategy='epoch',
    save_strategy='epoch',  # for tracking a progress
    logging_dir='./logs',
    load_best_model_at_end=True,  # choose the best epoch for an overfitting avoiding
    metric_for_best_model='f1',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

In [15]:
import time

start_time = time.perf_counter()

trainer.train()

end_time = time.perf_counter()
execution_time = end_time - start_time
print(f"Execution time: {execution_time:.6f} seconds")

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.054052,0.990329,0.976378,0.946565,0.961240
2,0.007600,0.050089,0.989362,0.968750,0.946565,0.957529
3,0.007600,0.056522,0.989362,0.968750,0.946565,0.957529


Execution time: 2117.099156 seconds


Epoch 1 has the best F1 score. 

F1 score for 2nd and 3rd epoch slightle worse. It's overfiiting: after epoch 1 model already learn patterns well.

# Step 6: Logistic Regression vs Transfomrmer

So now we have
|           | Baseline (LogReg) | Transformer (epoch 1, best) |
| :--- | :---: | :---: |
| Precision | 0.96 | 0.976 |
| Recall	| 0.92 | 0.947 |
| F1	    | 0.94 | 0.961 |

**The transformer beats the baseline on every metric.**

# Step 7: Tranformer Error Analysis

In [18]:
predictions = trainer.predict(test_dataset)
y_pred_transformer = np.argmax(predictions.predictions, axis=1)

# rebuild the same error analysis as baseline notebook
test_indices = y_test.index
errors_transformer = df.loc[test_indices].copy()
errors_transformer['predicted'] = y_pred_transformer
errors_transformer['true_label'] = y_test.values

false_positives_t = errors_transformer[(errors_transformer['label'] == 'ham') & (errors_transformer['predicted'] == 1)]
false_negatives_t = errors_transformer[(errors_transformer['label'] == 'spam') & (errors_transformer['predicted'] == 0)]

pd.set_option('display.max_colwidth', None)
print("FALSE POSITIVES:")
print(false_positives_t[['message']])
print("\nFALSE NEGATIVES:")
print(false_negatives_t[['message']])

FALSE POSITIVES:
                                                                                                                                                              message
3824  Please protect yourself from e-threats. SIB never asks for sensitive information like Passwords,ATM/SMS PIN thru email. Never share your password with anybody.
3736                                                                                                                                   It‘s £6 to get in, is that ok?
3044                                                                                                                       Your bill at 3 is £33.65 so thats not bad!

FALSE NEGATIVES:
                                                                                                                                                            message
1460                                                                              Bought one ringtone and now getting texts costing 3 pou

What we can see that the **transformer model catched quiet, conversational-style spam** that slipped baseline with numeric features.

Both models fails were **not caught ringtone-subscription spam**.

"RECPT 1/3. You have ordered a Ringtone. Your order is being processed..." => looks like a legitimate order confirmation.

"Your bill at 3 is £33.65 so thats not bad!" (FP in both models) => currency amount.

**"protect yourself from e-threats... never share your password"** => real message warning about fishing and containing vocabulary that looks like the phishing.